# 00 - Phenology & CSB Field Polygon Matching

This notebook:
1. Loads the cleaned Winter Wheat (Hard Red Winter) phenology scouting data
2. Loads USDA Crop Sequence Boundaries (CSB) 2013-2020
3. **Two matching methodologies:**
   - **Exact**: point falls within CSB polygon
   - **Buffer (300m)**: point buffered by 300m, then intersect with CSB polygon
4. Cross-checks CDL crop code per year vs our wheat observations
5. Exports matched field polygons for satellite data extraction

Results saved to separate folders: `processed/exact/` and `processed/buffer_300m/`

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
import fiona
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Paths
PHENOLOGY_PATH = 'data/raw/phenology/wheat_hrw_phenology_clean.parquet'
CSB_GDB_PATH = '/scratch/gautschi/vmangidi/WheatFlagLeaf/csb/NationalCSB_2013-2020_rev23/CSB1320.gdb'
CSB_FILTERED_PATH = '/scratch/gautschi/vmangidi/WheatFlagLeaf/csb/csb_target_states.gpkg'
OUTPUT_BASE = 'data/processed'

CDL_WINTER_WHEAT = 24
BUFFER_M = 300  # meters

CDL_NAMES = {
    1: 'Corn', 2: 'Cotton', 4: 'Sorghum', 5: 'Soybeans',
    21: 'Barley', 22: 'Durum Wheat', 23: 'Spring Wheat', 24: 'Winter Wheat',
    26: 'Dbl WinWht/Soybeans', 27: 'Rye', 28: 'Oats',
    29: 'Millet', 36: 'Alfalfa', 37: 'Other Hay',
    61: 'Fallow/Idle', 111: 'Open Water', 121: 'Developed/Open',
    131: 'Barren', 141: 'Deciduous Forest', 152: 'Shrubland',
    171: 'Grassland/Pasture', 176: 'Grassland/Pasture',
    225: 'Dbl WinWht/Corn', 226: 'Dbl WinWht/Sorghum', 236: 'Dbl WinWht/Cotton',
}

FIPS_TO_STATE = {
    '04':'AZ','05':'AR','06':'CA','08':'CO','17':'IL','18':'IN',
    '19':'IA','20':'KS','21':'KY','22':'LA','27':'MN','28':'MS',
    '29':'MO','30':'MT','31':'NE','35':'NM','38':'ND','39':'OH',
    '40':'OK','46':'SD','47':'TN','48':'TX','55':'WI','56':'WY'
}

## 1. Load Phenology Data

In [ ]:
df_pheno = pd.read_parquet(PHENOLOGY_PATH)
print(f'Phenology records: {len(df_pheno):,}')
print(f'Growing seasons: {sorted(df_pheno["growing_season"].unique())}')
print(f'\nGrowth stage counts:')
print(df_pheno['growth_stage'].value_counts().head(10))

geometry = [Point(lon, lat) for lat, lon in zip(df_pheno['lat'], df_pheno['lon'])]
gdf_pheno = gpd.GeoDataFrame(df_pheno, geometry=geometry, crs='EPSG:4326')
print(f'\nBounds: {gdf_pheno.total_bounds}')

## 2. Load CSB Polygons

In [ ]:
if os.path.exists(CSB_FILTERED_PATH):
    print(f'Loading pre-filtered CSB from {CSB_FILTERED_PATH}...')
    gdf_csb = gpd.read_file(CSB_FILTERED_PATH)
    print(f'Loaded {len(gdf_csb):,} CSB polygons')
else:
    min_lat, max_lat = gdf_pheno['lat'].min(), gdf_pheno['lat'].max()
    min_lon, max_lon = gdf_pheno['lon'].min(), gdf_pheno['lon'].max()
    print(f'Data extent: lat [{min_lat:.2f}, {max_lat:.2f}], lon [{min_lon:.2f}, {max_lon:.2f}]')

    # Pass 1: find which states overlap our data extent
    print('\nPass 1: Identifying states...')
    state_sample_pts = {}
    with fiona.open(CSB_GDB_PATH, layer='national1320') as src:
        crs_csb = src.crs
        for feat in src:
            fips = feat['properties']['STATEFIPS']
            if fips not in state_sample_pts:
                state_sample_pts[fips] = (feat['properties']['INSIDE_X'], feat['properties']['INSIDE_Y'])
                if len(state_sample_pts) >= 50:
                    break

    sample_gdf = gpd.GeoDataFrame(
        [{'fips': f, 'geometry': Point(x, y)} for f, (x, y) in state_sample_pts.items()],
        crs=crs_csb
    ).to_crs('EPSG:4326')

    BUF = 1.0
    candidate_fips = set(sample_gdf[
        (sample_gdf.geometry.y >= min_lat - BUF) & (sample_gdf.geometry.y <= max_lat + BUF) &
        (sample_gdf.geometry.x >= min_lon - BUF) & (sample_gdf.geometry.x <= max_lon + BUF)
    ]['fips'])
    print(f'States: {[FIPS_TO_STATE.get(f, f) for f in sorted(candidate_fips)]}')

    # Pass 2: read features for those states
    print(f'\nPass 2: Reading CSB features...')
    records = []
    with fiona.open(CSB_GDB_PATH, layer='national1320') as src:
        for i, feat in enumerate(src):
            if feat['properties']['STATEFIPS'] in candidate_fips:
                records.append(feat)
            if (i + 1) % 2_000_000 == 0:
                print(f'  Scanned {i+1:,}, kept {len(records):,}...')

    print(f'\nTotal CSB features: {len(records):,}')
    gdf_csb = gpd.GeoDataFrame.from_features(records, crs=crs_csb)
    gdf_csb.to_file(CSB_FILTERED_PATH, driver='GPKG')
    print(f'Saved to {CSB_FILTERED_PATH}')

gdf_csb['state'] = gdf_csb['STATEFIPS'].map(FIPS_TO_STATE)
print(f'\nCSB CRS: {gdf_csb.crs}')
print(f'States: {gdf_csb["state"].value_counts().to_dict()}')

## 3. Prepare for Spatial Join

In [ ]:
# Reproject points to CSB CRS (EPSG:5070, units = meters)
gdf_pheno_proj = gdf_pheno.to_crs(gdf_csb.crs)

# Pre-filter CSB to bounding box of our points
bounds = gdf_pheno_proj.total_bounds
pad = 10000  # 10km
gdf_csb_clip = gdf_csb.cx[bounds[0]-pad:bounds[2]+pad, bounds[1]-pad:bounds[3]+pad].copy()
print(f'CSB after bbox clip: {len(gdf_csb_clip):,} (from {len(gdf_csb):,})')

_ = gdf_csb_clip.sindex
print('Spatial index built.')

## 4. Method A: Exact Match (point within polygon)

In [ ]:
print('=' * 60)
print('METHOD A: EXACT MATCH (point within polygon)')
print('=' * 60)

gdf_exact = gpd.sjoin(gdf_pheno_proj, gdf_csb_clip, how='left', predicate='within')

if gdf_exact.index.duplicated().any():
    n_dupes = gdf_exact.index.duplicated().sum()
    print(f'Removing {n_dupes} duplicate matches')
    gdf_exact = gdf_exact[~gdf_exact.index.duplicated(keep='first')]

matched_a = gdf_exact['CSBID'].notna().sum()
total_a = len(gdf_exact)
print(f'\nMatched: {matched_a:,} / {total_a:,} ({matched_a/total_a*100:.1f}%)')
print(f'Unmatched: {total_a - matched_a:,}')
print(f'Unique CSB fields: {gdf_exact["CSBID"].nunique()}')

## 5. Method B: Buffer Match (300m)

300m buffer around each point, then intersect with CSB polygons.

In [ ]:
print('=' * 60)
print(f'METHOD B: BUFFER MATCH ({BUFFER_M}m buffer around points)')
print('=' * 60)

gdf_pheno_buf = gdf_pheno_proj.copy()
gdf_pheno_buf['geometry'] = gdf_pheno_buf.geometry.buffer(BUFFER_M)

gdf_buffer = gpd.sjoin(gdf_pheno_buf, gdf_csb_clip, how='left', predicate='intersects')

if gdf_buffer.index.duplicated().any():
    n_dupes = gdf_buffer.index.duplicated().sum()
    print(f'Removing {n_dupes} duplicate matches (keeping first)')
    gdf_buffer = gdf_buffer[~gdf_buffer.index.duplicated(keep='first')]

matched_b = gdf_buffer['CSBID'].notna().sum()
total_b = len(gdf_buffer)
print(f'\nMatched: {matched_b:,} / {total_b:,} ({matched_b/total_b*100:.1f}%)')
print(f'Unmatched: {total_b - matched_b:,}')
print(f'Unique CSB fields: {gdf_buffer["CSBID"].nunique()}')

print(f'\n--- Comparison ---')
print(f'Exact:  {matched_a:,} matched ({matched_a/total_a*100:.1f}%)')
print(f'Buffer: {matched_b:,} matched ({matched_b/total_b*100:.1f}%)')
print(f'Gained: {matched_b - matched_a:,} extra matches with {BUFFER_M}m buffer')

## 6. CDL Cross-Check (both methods)

In [ ]:
def cdl_crosscheck(gdf_joined, method_name):
    df = gdf_joined[gdf_joined['CSBID'].notna()].copy()
    
    df['cdl_crop'] = np.nan
    for year in range(2013, 2018):
        col = f'CDL{year}'
        if col in df.columns:
            mask = df['year'] == year
            df.loc[mask, 'cdl_crop'] = df.loc[mask, col]
    
    df['is_wheat_cdl'] = df['cdl_crop'] == CDL_WINTER_WHEAT
    
    print(f'\n{"="*60}')
    print(f'CDL CROSS-CHECK: {method_name}')
    print(f'{"="*60}')
    print(f'Matched observations: {len(df):,}')
    print(f'CDL = Winter Wheat: {df["is_wheat_cdl"].sum():,} ({df["is_wheat_cdl"].mean()*100:.1f}%)')
    print(f'CDL \u2260 Winter Wheat: {(~df["is_wheat_cdl"]).sum():,}')
    
    print(f'\nPer year:')
    for year in sorted(df['year'].unique()):
        sub = df[df['year'] == year]
        rate = sub['is_wheat_cdl'].mean() * 100
        print(f'  {year}: {sub["is_wheat_cdl"].sum():,} / {len(sub):,} ({rate:.1f}%)')
    
    non_wheat = df[~df['is_wheat_cdl']]
    print(f'\nTop non-wheat CDL codes:')
    for code, count in non_wheat['cdl_crop'].value_counts().head(10).items():
        name = CDL_NAMES.get(int(code), f'Code {int(code)}')
        print(f'  {int(code):>3d} ({name}): {count:,}')
    
    print(f'\nFlag leaf (CDL-confirmed wheat):')
    for stage in ['Flag Leaf Emerging', 'Flag Leaf Emerged']:
        sub = df[df['growth_stage'] == stage]
        if len(sub) > 0:
            rate = sub['is_wheat_cdl'].mean() * 100
            print(f'  {stage}: {sub["is_wheat_cdl"].sum():,} / {len(sub):,} ({rate:.1f}%)')
    
    return df

df_exact = cdl_crosscheck(gdf_exact, 'EXACT MATCH')
df_buffer = cdl_crosscheck(gdf_buffer, f'BUFFER {BUFFER_M}m')

## 7. Visualize Flag Leaf Timing

In [ ]:
df_viz = df_buffer[df_buffer['is_wheat_cdl']].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for stage in ['Flag Leaf Emerging', 'Flag Leaf Emerged']:
    sub = df_viz[df_viz['growth_stage'] == stage]
    axes[0].hist(sub['doy'], bins=30, alpha=0.6, label=stage)
axes[0].set_xlabel('Day of Year')
axes[0].set_ylabel('Count')
axes[0].set_title('Flag Leaf DOY Distribution')
axes[0].legend()

for gs in sorted(df_viz['growing_season'].unique()):
    sub = df_viz[(df_viz['growing_season'] == gs) & (df_viz['growth_stage'] == 'Flag Leaf Emerging')]
    if len(sub) > 10:
        axes[1].hist(sub['dos'], bins=20, alpha=0.5, label=gs)
axes[1].set_xlabel('Day of Season (from Jul 1)')
axes[1].set_ylabel('Count')
axes[1].set_title('Flag Leaf Emerging per Season')
axes[1].legend()

for stage, marker in zip(['Flag Leaf Emerging', 'Flag Leaf Emerged'], ['o', 's']):
    sub = df_viz[df_viz['growth_stage'] == stage]
    axes[2].scatter(sub['lat'], sub['doy'], alpha=0.15, s=10, marker=marker, label=stage)
axes[2].set_xlabel('Latitude')
axes[2].set_ylabel('Day of Year')
axes[2].set_title('Flag Leaf Timing vs Latitude')
axes[2].legend()

plt.tight_layout()
plt.show()

## 8. Full Phenological Cycle

In [ ]:
stage_order = {
    'Seed': 1, 'Germinating': 2, 'Emerging': 3, 'Emerging - Seedling': 3,
    'Seedling': 4, 'Seedling - 1 Leaf': 4,
    '1 Leaf': 5, '2 Leaf': 6, '3 Leaf': 7,
    'Begin Tillering': 8, '1-2 Tiller': 9, '2-4 Tiller': 10, '4-6 Tiller': 11,
    '6-8 Tiller': 12, '8+ Tiller': 13,
    'Spring Vegetative': 14, 'Dormancy': 15,
    'Jointing': 16, '1st Node Visible': 17, '2nd Node Visible': 18, '3rd Node Visible': 19,
    'Flag Leaf Emerging': 20, 'Flag Leaf Emerged': 21,
    'Early Boot': 22, 'Boot': 23,
    'Head Emerging': 24, 'Heading': 25,
    'Early Bloom': 26, 'Bloom': 27,
    'Milk': 28, '1/4 Berry': 29, '1/2 Berry': 30, '3/4 Berry': 31,
    'Soft Dough': 32, 'Medium Dough': 33, 'Hard Dough': 34,
    'Maturity': 35, 'Harvest Ready': 36
}

df_viz['stage_num'] = df_viz['growth_stage'].map(stage_order)
stage_timing = df_viz.groupby('growth_stage').agg(
    stage_num=('stage_num', 'first'),
    median_dos=('dos', 'median'),
    q25_dos=('dos', lambda x: x.quantile(0.25)),
    q75_dos=('dos', lambda x: x.quantile(0.75)),
    count=('dos', 'count')
).dropna(subset=['stage_num']).sort_values('stage_num')

fig, ax = plt.subplots(figsize=(14, 8))
colors = ['red' if 'Flag' in s else 'steelblue' for s in stage_timing.index]
ax.barh(range(len(stage_timing)),
        stage_timing['q75_dos'] - stage_timing['q25_dos'],
        left=stage_timing['q25_dos'], color=colors, alpha=0.7)
ax.scatter(stage_timing['median_dos'], range(len(stage_timing)), color='black', s=20, zorder=5)
ax.set_yticks(range(len(stage_timing)))
ax.set_yticklabels([f'{s} (n={int(stage_timing.loc[s,"count"]):,})' for s in stage_timing.index])
ax.set_xlabel('Day of Season (from Jul 1)')
ax.set_title('Winter Wheat Phenological Cycle (IQR + Median)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Export Results (both methods)

In [ ]:
def export_results(df_matched, gdf_csb, method_name, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    # Keep ALL matched observations regardless of CDL crop code
    all_csbids = df_matched['CSBID'].unique()
    gdf_fields = gdf_csb[gdf_csb['CSBID'].isin(all_csbids)].copy()
    
    fields_path = os.path.join(output_dir, 'wheat_hrw_field_polygons.gpkg')
    gdf_fields.to_file(fields_path, driver='GPKG')
    
    cols = ['date', 'year', 'doy', 'dos', 'month', 'growing_season',
            'lat', 'lon', 'growth_stage', 'crop_condition',
            'CSBID', 'CSBACRES', 'cdl_crop', 'is_wheat_cdl',
            'CDL2013', 'CDL2014', 'CDL2015', 'CDL2016', 'CDL2017',
            'STATEFIPS', 'CNTY']
    cols_available = [c for c in cols if c in df_matched.columns]
    matched_path = os.path.join(output_dir, 'wheat_hrw_phenology_csb_matched.parquet')
    df_matched[cols_available].to_parquet(matched_path, index=False)
    
    n_wheat = df_matched['is_wheat_cdl'].sum()
    print(f'\n--- {method_name} ---')
    print(f'  Total observations: {len(df_matched):,}')
    print(f'  Unique fields (ALL): {len(all_csbids):,}')
    print(f'  Total area: {gdf_fields["CSBACRES"].sum():,.0f} acres')
    print(f'  CDL = wheat: {n_wheat:,} ({n_wheat/len(df_matched)*100:.1f}%)')
    print(f'  Flag Leaf Emerging: {len(df_matched[df_matched["growth_stage"]=="Flag Leaf Emerging"]):,}')
    print(f'  Flag Leaf Emerged:  {len(df_matched[df_matched["growth_stage"]=="Flag Leaf Emerged"]):,}')
    print(f'  Polygons: {fields_path}')
    print(f'  Data: {matched_path}')
    return df_matched, gdf_fields

res_exact, fields_exact = export_results(
    df_exact, gdf_csb, 'EXACT', os.path.join(OUTPUT_BASE, 'exact'))

res_buffer, fields_buffer = export_results(
    df_buffer, gdf_csb, f'BUFFER {BUFFER_M}m', os.path.join(OUTPUT_BASE, f'buffer_{BUFFER_M}m'))

In [ ]:
print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'Total phenology observations: {len(df_pheno):,}')
print()
print(f'{"Metric":<40} {"Exact":>10} {"Buffer":>10}')
print(f'{"-"*40} {"-"*10} {"-"*10}')
print(f'{"Matched to CSB polygon":<40} {matched_a:>10,} {matched_b:>10,}')
print(f'{"Unique fields (ALL)":<40} {res_exact["CSBID"].nunique():>10,} {res_buffer["CSBID"].nunique():>10,}')
print(f'{"CDL = wheat (info only)":<40} {res_exact["is_wheat_cdl"].sum():>10,} {res_buffer["is_wheat_cdl"].sum():>10,}')
n_ex_emerging = len(res_exact[res_exact['growth_stage']=='Flag Leaf Emerging'])
n_bu_emerging = len(res_buffer[res_buffer['growth_stage']=='Flag Leaf Emerging'])
n_ex_emerged = len(res_exact[res_exact['growth_stage']=='Flag Leaf Emerged'])
n_bu_emerged = len(res_buffer[res_buffer['growth_stage']=='Flag Leaf Emerged'])
print(f'{"Flag Leaf Emerging":<40} {n_ex_emerging:>10,} {n_bu_emerging:>10,}')
print(f'{"Flag Leaf Emerged":<40} {n_ex_emerged:>10,} {n_bu_emerged:>10,}')
print()
print(f'Outputs (ALL matched fields, no CDL filter):')
print(f'  Exact:  {OUTPUT_BASE}/exact/')
print(f'  Buffer: {OUTPUT_BASE}/buffer_{BUFFER_M}m/')